# Competição Kaggle 1 — Classificação da Coluna Vertebral

**Aprendizado de Máquina — Prof. Me. Otávio Parraga**

**Integrantes:** Estevam Cabral

Classes: `Hernia`, `Spondylolisthesis`, `Normal`.
Modelos exigidos: KNN, Naïve Bayes e Árvore de Decisão.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

SEED = 42
np.random.seed(SEED)

In [ ]:
train_df = pd.read_csv('kaggle_comp_1/train.csv', index_col=[0])
test_df = pd.read_csv('kaggle_comp_1/test.csv', index_col=[0])

print(train_df.shape, test_df.shape)
train_df.head()

In [ ]:
print(train_df['class'].value_counts())
print()
print(train_df.describe().T)
print()
print('Nulos no treino:', train_df.isna().sum().sum(), '| Nulos no teste:', test_df.isna().sum().sum())

In [ ]:
X = train_df.drop(columns=['class'])
y = train_df['class']

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        ds = X['degree_spondylolisthesis'].clip(-20, 120)
        X['ds_clip'] = ds
        X['ds_log'] = np.sign(ds) * np.log1p(np.abs(ds))
        X['tilt_ratio'] = (X['pelvic_tilt'] / X['pelvic_incidence'].replace(0, np.nan)).fillna(0)
        X['ll_ss'] = X['lumbar_lordosis_angle'] - X['sacral_slope']
        return X.drop(columns=['pelvic_incidence'])

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

modelos = {
    'KNN': (
        Pipeline([
            ('fe', FeatureEngineer()),
            ('sc', PowerTransformer()),
            ('clf', KNeighborsClassifier()),
        ]),
        {
            'clf__n_neighbors': list(range(1, 42, 2)),
            'clf__weights': ['uniform', 'distance'],
            'clf__p': [1, 2],
        },
    ),
    'NaiveBayes': (
        Pipeline([
            ('fe', FeatureEngineer()),
            ('sc', PowerTransformer()),
            ('clf', GaussianNB()),
        ]),
        {
            'clf__var_smoothing': np.logspace(-11, 0, 40),
        },
    ),
    'DecisionTree': (
        Pipeline([
            ('fe', FeatureEngineer()),
            ('clf', DecisionTreeClassifier(random_state=SEED)),
        ]),
        {
            'clf__criterion': ['gini', 'entropy'],
            'clf__max_depth': [2, 3, 4, 5, 6, 8, None],
            'clf__min_samples_leaf': [1, 2, 4, 8, 12],
            'clf__min_samples_split': [2, 5, 10],
            'clf__ccp_alpha': [0.0, 0.005, 0.01, 0.02],
        },
    ),
}

In [ ]:
resultados = {}

for nome, (pipe, grid) in modelos.items():
    busca = GridSearchCV(pipe, grid, cv=cv, scoring='accuracy', n_jobs=-1, refit=True)
    busca.fit(X, y)
    resultados[nome] = busca
    print(f'{nome:13s} | acuracia CV = {busca.best_score_:.4f}')
    print(f'{"":13s} | melhores params = {busca.best_params_}')
    print()

In [ ]:
melhor_nome = max(resultados, key=lambda k: resultados[k].best_score_)
melhor_modelo = resultados[melhor_nome].best_estimator_

print('Modelo escolhido:', melhor_nome)
print('Acuracia CV:', round(resultados[melhor_nome].best_score_, 4))

In [ ]:
scores = cross_val_score(melhor_modelo, X, y, cv=cv, scoring='accuracy')
print('Acuracia por fold:', np.round(scores, 4))
print('Media: %.4f  |  Desvio: %.4f' % (scores.mean(), scores.std()))

In [ ]:
pred_oof = cross_val_predict(clone(melhor_modelo), X, y, cv=cv, n_jobs=-1)

print('Accuracy :', accuracy_score(y, pred_oof))
print('F1       :', f1_score(y, pred_oof, average='macro'))
print('Precision:', precision_score(y, pred_oof, average='macro', zero_division=0))
print('Recall   :', recall_score(y, pred_oof, average='macro'))
print()
print(classification_report(y, pred_oof, zero_division=0))
print(pd.DataFrame(
    confusion_matrix(y, pred_oof, labels=sorted(y.unique())),
    index=['real_' + c for c in sorted(y.unique())],
    columns=['pred_' + c for c in sorted(y.unique())],
))

In [ ]:
melhor_modelo.fit(X, y)
final_predictions = melhor_modelo.predict(test_df)

print(pd.Series(final_predictions).value_counts())

In [ ]:
def create_submission_file(predictions, test_df, submission_file_name="submission.csv"):
    submission_df = pd.DataFrame({'id': test_df.index, 'Target': predictions})
    submission_df.to_csv(submission_file_name, index=False)
    print(f"Submission file '{submission_file_name}' created successfully.")

create_submission_file(final_predictions, test_df)

In [ ]:
pd.read_csv('submission.csv').head()